# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás diseñado un manifiesto para el caso de calidad del vino y una lista de invariantes que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [ ]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
+
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | ______________________________ |
| Orden y nombres de features | ______________________________ | ______________________________ |
| Versión del preprocesado | ______________________________ | ______________________________ |
| Etiquetas de salida permitidas | ______________________________ | ______________________________ |
| Versión del modelo | ______________________________ | ______________________________ |

## 3. Proponer el manifiesto

Rellena el siguiente borrador. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [ ]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

manifest_draft = {
    "schema_version": "TODO",
    "model_version": "TODO",
    "preprocessing_version": "TODO",
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": "TODO",
}

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nSustituye los valores TODO por decisiones justificadas en la celda anterior.")

## 4. Contrato de almacenamiento

Elige una estructura de directorios para que una carpeta represente un bundle completo. Una propuesta razonable es:

~~~text
models/
└── wine_quality_bundle/
    ├── manifest.json
    └── model.joblib
~~~

Escribe en tus propias palabras las tres garantías que debe proporcionar **save_model_bundle**:

- ______________________________________________________________________
- ______________________________________________________________________
- ______________________________________________________________________

Pista: piensa en consistencia, compatibilidad y qué ocurre si el proceso se interrumpe mientras escribe.

## 5. Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.